# Ejercicios Prácticos: Redes Generativas Adversarias (GANs)

Este notebook contiene ejercicios graduales para aplicar y consolidar los conceptos de GANs.

**Estructura:**
- Parte 1: Fundamentos y Arquitectura
- Parte 2: Implementación Básica
- Parte 3: Entrenamiento y Diagnóstico
- Parte 4: Variantes Avanzadas
- Parte 5: Evaluación y Métricas

In [ ]:
# Importaciones necesarias
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuración
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

---
## PARTE 1: Fundamentos y Arquitectura

### Ejercicio 1.1: Entendiendo el Espacio Latente

**Objetivo:** Explorar cómo el vector latente z controla la generación.

In [ ]:
# EJERCICIO 1.1
# TODO: Completa las siguientes funciones

def sample_latent_vectors(n_samples, latent_dim, distribution='normal'):
    """
    Genera n_samples vectores latentes de dimensión latent_dim.
    
    Args:
        n_samples: número de vectores a generar
        latent_dim: dimensionalidad del espacio latente
        distribution: 'normal' o 'uniform'
    
    Returns:
        Tensor de forma (n_samples, latent_dim)
    """
    # TODO: Implementa el muestreo de vectores latentes
    # Si distribution='normal', usa distribución normal estándar
    # Si distribution='uniform', usa uniforme en [-1, 1]
    pass

def interpolate_latent(z1, z2, num_steps=10):
    """
    Crea una interpolación lineal entre dos vectores latentes.
    
    Args:
        z1: primer vector latente
        z2: segundo vector latente
        num_steps: número de pasos en la interpolación
    
    Returns:
        Tensor de forma (num_steps, latent_dim)
    """
    # TODO: Implementa interpolación lineal
    # z(t) = (1-t)*z1 + t*z2, donde t va de 0 a 1
    pass

# Prueba tus funciones
z_samples = sample_latent_vectors(5, 100, 'normal')
print(f'Forma de z_samples: {z_samples.shape}')
print(f'Media: {z_samples.mean():.4f}, Std: {z_samples.std():.4f}')

z1 = torch.randn(1, 100)
z2 = torch.randn(1, 100)
interpolation = interpolate_latent(z1, z2, 10)
print(f'\nForma de interpolación: {interpolation.shape}')

### Ejercicio 1.2: Construyendo un Generador Simple

**Objetivo:** Implementar un generador básico para imágenes de 28x28.

In [ ]:
# EJERCICIO 1.2

class SimpleGenerator(nn.Module):
    def __init__(self, latent_dim=100, img_size=28, channels=1):
        super(SimpleGenerator, self).__init__()
        self.img_size = img_size
        self.channels = channels
        
        # TODO: Define la arquitectura del generador
        # Sugerencia: Usa capas lineales con BatchNorm y LeakyReLU
        # Estructura recomendada:
        # latent_dim -> 256 -> 512 -> 1024 -> img_size*img_size*channels
        # Usa Tanh() al final para salida en [-1, 1]
        
        self.model = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, z):
        # TODO: Implementa el forward pass
        # Recuerda hacer reshape al final para obtener (batch, channels, img_size, img_size)
        pass

# Prueba tu generador
gen = SimpleGenerator(latent_dim=100, img_size=28, channels=1)
z_test = torch.randn(4, 100)
fake_imgs = gen(z_test)
print(f'Forma de salida del generador: {fake_imgs.shape}')
print(f'Rango de valores: [{fake_imgs.min():.2f}, {fake_imgs.max():.2f}]')

### Ejercicio 1.3: Construyendo un Discriminador Simple

**Objetivo:** Implementar un discriminador básico.

In [ ]:
# EJERCICIO 1.3

class SimpleDiscriminator(nn.Module):
    def __init__(self, img_size=28, channels=1):
        super(SimpleDiscriminator, self).__init__()
        
        # TODO: Define la arquitectura del discriminador
        # Estructura recomendada:
        # img_size*img_size*channels -> 512 -> 256 -> 1
        # Usa LeakyReLU(0.2) y NO uses BatchNorm en la primera capa
        # Usa Sigmoid() al final para probabilidad en [0, 1]
        
        self.model = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, img):
        # TODO: Implementa el forward pass
        # Recuerda aplanar la imagen primero
        pass

# Prueba tu discriminador
disc = SimpleDiscriminator(img_size=28, channels=1)
test_imgs = torch.randn(4, 1, 28, 28)
predictions = disc(test_imgs)
print(f'Forma de salida del discriminador: {predictions.shape}')
print(f'Predicciones: {predictions.squeeze()}')

---
## PARTE 2: Implementación Básica

### Ejercicio 2.1: Calculando las Pérdidas

**Objetivo:** Implementar las funciones de pérdida para G y D.

In [ ]:
# EJERCICIO 2.1

def discriminator_loss(real_output, fake_output, criterion):
    """
    Calcula la pérdida del discriminador.
    
    Args:
        real_output: salida del discriminador para imágenes reales
        fake_output: salida del discriminador para imágenes falsas
        criterion: función de pérdida (BCELoss)
    
    Returns:
        Pérdida total del discriminador
    """
    # TODO: Implementa la pérdida del discriminador
    # Recuerda: D quiere maximizar log(D(x)) + log(1 - D(G(z)))
    # En la práctica, minimizamos la pérdida negativa
    pass

def generator_loss(fake_output, criterion, use_non_saturating=True):
    """
    Calcula la pérdida del generador.
    
    Args:
        fake_output: salida del discriminador para imágenes falsas
        criterion: función de pérdida (BCELoss)
        use_non_saturating: si True, usa la versión no saturante
    
    Returns:
        Pérdida del generador
    """
    # TODO: Implementa la pérdida del generador
    # Pérdida original: minimizar log(1 - D(G(z)))
    # Non-saturating: maximizar log(D(G(z)))
    pass

# Prueba tus funciones de pérdida
criterion = nn.BCELoss()
real_out = torch.tensor([[0.9], [0.8], [0.95]])
fake_out = torch.tensor([[0.1], [0.2], [0.15]])

d_loss = discriminator_loss(real_out, fake_out, criterion)
g_loss = generator_loss(fake_out, criterion)

print(f'Pérdida del Discriminador: {d_loss:.4f}')
print(f'Pérdida del Generador: {g_loss:.4f}')

### Ejercicio 2.2: Loop de Entrenamiento

**Objetivo:** Implementar el loop de entrenamiento alternado.

In [ ]:
# EJERCICIO 2.2

def train_gan_step(generator, discriminator, real_imgs, 
                   optimizer_G, optimizer_D, criterion, latent_dim, device):
    """
    Realiza un paso de entrenamiento de la GAN.
    
    Args:
        generator: modelo del generador
        discriminator: modelo del discriminador
        real_imgs: batch de imágenes reales
        optimizer_G: optimizador del generador
        optimizer_D: optimizador del discriminador
        criterion: función de pérdida
        latent_dim: dimensión del espacio latente
        device: dispositivo (cuda/cpu)
    
    Returns:
        d_loss, g_loss: pérdidas del discriminador y generador
    """
    batch_size = real_imgs.size(0)
    real_imgs = real_imgs.to(device)
    
    # Etiquetas
    real_labels = torch.ones(batch_size, 1, device=device)
    fake_labels = torch.zeros(batch_size, 1, device=device)
    
    # TODO: Paso 1 - Entrenar el Discriminador
    # 1. Calcular pérdida con imágenes reales
    # 2. Generar imágenes falsas
    # 3. Calcular pérdida con imágenes falsas (usar .detach() para no backprop al generador)
    # 4. Sumar pérdidas y hacer backward
    # 5. Actualizar pesos del discriminador
    
    optimizer_D.zero_grad()
    # TODO: Completa aquí
    
    # TODO: Paso 2 - Entrenar el Generador
    # 1. Generar nuevas imágenes falsas
    # 2. Obtener predicciones del discriminador (sin .detach() ahora)
    # 3. Calcular pérdida del generador
    # 4. Hacer backward y actualizar pesos
    
    optimizer_G.zero_grad()
    # TODO: Completa aquí
    
    return d_loss.item(), g_loss.item()

### Ejercicio 2.3: Entrenamiento Completo en MNIST

**Objetivo:** Entrenar una GAN simple en MNIST.

In [ ]:
# EJERCICIO 2.3

# Configuración
latent_dim = 100
lr = 0.0002
batch_size = 64
num_epochs = 5  # Usa más épocas para mejores resultados

# Cargar datos
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Crear modelos
generator = SimpleGenerator(latent_dim=latent_dim).to(device)
discriminator = SimpleDiscriminator().to(device)

# Optimizadores
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Función de pérdida
criterion = nn.BCELoss()

# Vector fijo para visualización
fixed_noise = torch.randn(16, latent_dim, device=device)

# Listas para guardar pérdidas
d_losses = []
g_losses = []

# TODO: Implementa el loop de entrenamiento completo
for epoch in range(num_epochs):
    epoch_d_loss = 0
    epoch_g_loss = 0
    
    for i, (real_imgs, _) in enumerate(dataloader):
        # TODO: Llama a train_gan_step aquí
        pass
    
    # TODO: Calcula pérdidas promedio de la época
    # TODO: Imprime el progreso
    # TODO: Genera y guarda imágenes con fixed_noise cada época

# Graficar pérdidas
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label='Discriminador')
plt.plot(g_losses, label='Generador')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.title('Pérdidas durante el entrenamiento')
plt.show()

---
## PARTE 3: Entrenamiento y Diagnóstico

### Ejercicio 3.1: Detectando Mode Collapse

**Objetivo:** Implementar métricas para detectar mode collapse.

In [ ]:
# EJERCICIO 3.1

def calculate_diversity_score(generator, latent_dim, n_samples=1000, device='cpu'):
    """
    Calcula un score de diversidad basado en la varianza de las imágenes generadas.
    
    Args:
        generator: modelo del generador
        latent_dim: dimensión del espacio latente
        n_samples: número de muestras a generar
        device: dispositivo
    
    Returns:
        diversity_score: score de diversidad (mayor = más diverso)
    """
    generator.eval()
    
    # TODO: Implementa el cálculo de diversidad
    # Sugerencia: Genera n_samples imágenes y calcula la varianza promedio
    # También puedes calcular distancias entre pares de imágenes
    
    with torch.no_grad():
        pass
    
    generator.train()
    return diversity_score

def detect_mode_collapse(generator, latent_dim, threshold=0.1, device='cpu'):
    """
    Detecta si hay mode collapse comparando imágenes generadas.
    
    Returns:
        True si se detecta mode collapse
    """
    # TODO: Implementa detección de mode collapse
    # Genera varias imágenes y compara su similitud
    pass

### Ejercicio 3.2: Visualización del Espacio Latente

**Objetivo:** Explorar cómo las dimensiones del espacio latente afectan la generación.

In [ ]:
# EJERCICIO 3.2

def explore_latent_dimension(generator, dim_to_vary, values, base_z=None, device='cpu'):
    """
    Explora el efecto de variar una dimensión específica del espacio latente.
    
    Args:
        generator: modelo entrenado
        dim_to_vary: índice de la dimensión a variar
        values: lista de valores para esa dimensión
        base_z: vector latente base (si None, se genera aleatorio)
        device: dispositivo
    
    Returns:
        Tensor con las imágenes generadas
    """
    generator.eval()
    
    if base_z is None:
        base_z = torch.randn(1, generator.latent_dim, device=device)
    
    # TODO: Genera imágenes variando solo la dimensión especificada
    images = []
    with torch.no_grad():
        for val in values:
            # Copia base_z y modifica solo dim_to_vary
            pass
    
    generator.train()
    return torch.cat(images, dim=0)

# TODO: Usa esta función para explorar diferentes dimensiones
# Visualiza los resultados con make_grid

### Ejercicio 3.3: Interpolación en el Espacio Latente

**Objetivo:** Crear transiciones suaves entre imágenes.

In [ ]:
# EJERCICIO 3.3

def create_interpolation_grid(generator, n_rows=5, n_cols=10, device='cpu'):
    """
    Crea una cuadrícula de interpolaciones.
    
    Args:
        generator: modelo entrenado
        n_rows: número de filas (diferentes pares de puntos)
        n_cols: número de columnas (pasos de interpolación)
    
    Returns:
        Grid de imágenes interpoladas
    """
    generator.eval()
    
    # TODO: Crea n_rows interpolaciones diferentes
    all_images = []
    
    with torch.no_grad():
        for row in range(n_rows):
            # Genera dos puntos aleatorios
            # Crea interpolación entre ellos
            # Genera imágenes para cada punto intermedio
            pass
    
    generator.train()
    # TODO: Organiza las imágenes en un grid y visualiza
    return grid

# Prueba tu función
# grid = create_interpolation_grid(generator)
# plt.figure(figsize=(15, 10))
# plt.imshow(grid)
# plt.axis('off')
# plt.show()

---
## PARTE 4: Variantes Avanzadas

### Ejercicio 4.1: Implementando DCGAN

**Objetivo:** Crear versiones convolucionales del generador y discriminador.

In [ ]:
# EJERCICIO 4.1

class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=100, ngf=64):
        super(DCGANGenerator, self).__init__()
        
        # TODO: Implementa el generador DCGAN
        # Arquitectura sugerida:
        # latent_dim x 1 x 1 -> ngf*8 x 4 x 4 -> ngf*4 x 8 x 8 -> 
        # ngf*2 x 16 x 16 -> ngf x 32 x 32 -> 1 x 64 x 64
        # Usa ConvTranspose2d, BatchNorm2d, ReLU, y Tanh al final
        
        self.main = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, z):
        # z debe tener forma (batch, latent_dim, 1, 1)
        return self.main(z)

class DCGANDiscriminator(nn.Module):
    def __init__(self, ndf=64):
        super(DCGANDiscriminator, self).__init__()
        
        # TODO: Implementa el discriminador DCGAN
        # Usa Conv2d, BatchNorm2d (excepto primera capa), LeakyReLU(0.2)
        # y Sigmoid al final
        
        self.main = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, img):
        return self.main(img).view(-1, 1)

# TODO: Aplica weights_init a ambas redes
def weights_init(m):
    classname = m.__class__.__name__
    # TODO: Implementa inicialización según las guías de DCGAN
    pass

### Ejercicio 4.2: Conditional GAN (CGAN)

**Objetivo:** Modificar la GAN para generar dígitos específicos.

In [ ]:
# EJERCICIO 4.2

class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=100, n_classes=10):
        super(ConditionalGenerator, self).__init__()
        
        # TODO: Implementa un generador condicional
        # La entrada será [z, one_hot_label] concatenados
        # Por ejemplo: latent_dim + n_classes -> ...
        
        self.label_embedding = nn.Embedding(n_classes, n_classes)
        
        self.model = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, z, labels):
        # TODO: Concatena z con el embedding de labels
        # y pasa por el modelo
        pass

class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_classes=10):
        super(ConditionalDiscriminator, self).__init__()
        
        # TODO: Implementa un discriminador condicional
        # La entrada será la imagen con un canal extra para la clase
        
        self.label_embedding = nn.Embedding(n_classes, 28*28)
        
        self.model = nn.Sequential(
            # TODO: Completa aquí
        )
    
    def forward(self, img, labels):
        # TODO: Concatena imagen con embedding de labels
        pass

# TODO: Modifica el loop de entrenamiento para usar labels

### Ejercicio 4.3: Implementando Progressive Growing

**Objetivo:** Implementar crecimiento progresivo de resolución.

In [ ]:
# EJERCICIO 4.3

class ProgressiveGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super(ProgressiveGenerator, self).__init__()
        
        # TODO: Implementa bloques para diferentes resoluciones
        # Cada bloque aumenta la resolución 2x
        
        self.initial_block = self._make_initial_block(latent_dim)
        self.blocks = nn.ModuleList([
            # TODO: Añade bloques para 8x8, 16x16, 32x32, etc.
        ])
        
        self.to_rgb_layers = nn.ModuleList([
            # TODO: Capas para convertir features a RGB en cada resolución
        ])
    
    def _make_initial_block(self, latent_dim):
        # TODO: Bloque inicial (latent -> 4x4)
        pass
    
    def forward(self, z, depth, alpha):
        """
        Args:
            z: vector latente
            depth: nivel de profundidad actual (0=4x4, 1=8x8, etc.)
            alpha: factor de transición suave [0, 1]
        """
        # TODO: Implementa forward con transición suave
        # Cuando alpha < 1, mezcla salida de resolución anterior con actual
        pass

# TODO: Implementa ProgressiveDiscriminator similar

---
## PARTE 5: Evaluación y Métricas

### Ejercicio 5.1: Calculando Inception Score

**Objetivo:** Implementar el cálculo de IS desde cero.

In [ ]:
# EJERCICIO 5.1

from torchvision import models
from scipy.stats import entropy

def calculate_inception_score(generator, n_samples=50000, batch_size=32, 
                              splits=10, device='cpu'):
    """
    Calcula el Inception Score de imágenes generadas.
    
    Args:
        generator: modelo del generador
        n_samples: número de imágenes a generar
        batch_size: tamaño del batch
        splits: número de splits para el cálculo
    
    Returns:
        inception_score: IS calculado
        std: desviación estándar entre splits
    """
    # TODO: Carga Inception-v3 preentrenado
    inception_model = models.inception_v3(pretrained=True, transform_input=False)
    inception_model.eval()
    inception_model.to(device)
    
    # TODO: Genera imágenes y obtén predicciones
    preds = []
    generator.eval()
    
    with torch.no_grad():
        for i in range(0, n_samples, batch_size):
            # Genera imágenes
            # Redimensiona a 299x299 para Inception
            # Obtén predicciones de clases
            pass
    
    # TODO: Calcula IS usando la fórmula
    # IS = exp(E[KL(p(y|x) || p(y))])
    preds = np.concatenate(preds, axis=0)
    
    scores = []
    for i in range(splits):
        # Divide predicciones en splits
        part = preds[i * (n_samples // splits): (i + 1) * (n_samples // splits), :]
        # Calcula p(y) marginal
        # Calcula KL divergence
        pass
    
    return np.mean(scores), np.std(scores)

### Ejercicio 5.2: Calculando FID Score

**Objetivo:** Implementar el cálculo de FID.

In [ ]:
# EJERCICIO 5.2

from scipy import linalg

def calculate_fid_score(generator, real_dataloader, n_samples=10000, 
                       batch_size=50, device='cpu'):
    """
    Calcula el Fréchet Inception Distance.
    
    Args:
        generator: modelo del generador
        real_dataloader: dataloader con imágenes reales
        n_samples: número de muestras a usar
    
    Returns:
        fid_score: FID calculado
    """
    # TODO: Carga Inception-v3 sin la capa FC final
    inception_model = models.inception_v3(pretrained=True, transform_input=False)
    inception_model.fc = nn.Identity()
    inception_model.eval()
    inception_model.to(device)
    
    def extract_features(dataloader, n_samples):
        """Extrae features de Inception"""
        features = []
        # TODO: Implementa extracción de features
        return np.concatenate(features, axis=0)
    
    # TODO: Extrae features de imágenes reales
    real_features = extract_features(real_dataloader, n_samples)
    
    # TODO: Genera imágenes falsas y extrae features
    fake_features = []
    generator.eval()
    with torch.no_grad():
        for i in range(0, n_samples, batch_size):
            # Genera imágenes
            # Extrae features
            pass
    fake_features = np.concatenate(fake_features, axis=0)
    
    # TODO: Calcula estadísticas (media y covarianza)
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    
    mu_fake = np.mean(fake_features, axis=0)
    sigma_fake = np.cov(fake_features, rowvar=False)
    
    # TODO: Calcula FID usando la fórmula
    # FID = ||mu_r - mu_g||^2 + Tr(sigma_r + sigma_g - 2*sqrt(sigma_r * sigma_g))
    diff = mu_real - mu_fake
    covmean, _ = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)
    
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    
    return fid

### Ejercicio 5.3: Dashboard de Evaluación Completo

**Objetivo:** Crear un dashboard para monitorear todas las métricas.

In [ ]:
# EJERCICIO 5.3

class GANEvaluator:
    def __init__(self, generator, discriminator, real_dataloader, device='cpu'):
        self.generator = generator
        self.discriminator = discriminator
        self.real_dataloader = real_dataloader
        self.device = device
        
        self.history = {
            'd_loss': [],
            'g_loss': [],
            'diversity': [],
            'fid': [],
            'is': []
        }
    
    def evaluate_all(self, epoch):
        """Evalúa todas las métricas"""
        # TODO: Calcula todas las métricas
        # - Diversity score
        # - FID (cada N épocas)
        # - IS (cada N épocas)
        # - Visualizaciones
        pass
    
    def plot_metrics(self):
        """Grafica todas las métricas"""
        # TODO: Crea un dashboard con múltiples subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        # Pérdidas
        axes[0, 0].plot(self.history['d_loss'], label='D Loss')
        axes[0, 0].plot(self.history['g_loss'], label='G Loss')
        axes[0, 0].set_title('Pérdidas')
        axes[0, 0].legend()
        
        # TODO: Añade más plots para diversity, FID, IS, etc.
        # También añade visualizaciones de imágenes generadas
        
        plt.tight_layout()
        plt.show()
    
    def generate_report(self):
        """Genera un reporte textual completo"""
        # TODO: Imprime estadísticas resumidas
        pass

---
## PROYECTOS FINALES

### Proyecto 1: GAN para Generación de Rostros

**Objetivo:** Entrenar una DCGAN en el dataset CelebA para generar rostros realistas.

**Tareas:**
1. Descarga y preprocesa el dataset CelebA
2. Implementa DCGAN con resolución de al menos 64x64
3. Entrena por suficientes épocas para obtener resultados de calidad
4. Implementa exploración del espacio latente
5. Calcula FID e IS finales
6. Crea visualizaciones atractivas de los resultados

In [ ]:
# PROYECTO 1
# TODO: Tu implementación aquí

### Proyecto 2: Conditional GAN para Control Fino

**Objetivo:** Crear una CGAN que permita generar dígitos MNIST específicos con alta fidelidad.

**Tareas:**
1. Implementa CGAN con arquitectura mejorada
2. Entrena hasta obtener dígitos de alta calidad
3. Implementa interfaz para generar dígitos específicos
4. Explora interpolación entre diferentes clases
5. Analiza qué aprende cada capa del generador

In [ ]:
# PROYECTO 2
# TODO: Tu implementación aquí

### Proyecto 3: Comparación de Variantes de GANs

**Objetivo:** Comparar empíricamente diferentes arquitecturas y técnicas de entrenamiento.

**Tareas:**
1. Implementa al menos 3 variantes (vanilla GAN, DCGAN, WGAN-GP)
2. Entrena todas con el mismo dataset
3. Compara usando múltiples métricas (FID, IS, diversidad)
4. Analiza estabilidad del entrenamiento
5. Crea un reporte comparativo detallado con visualizaciones

In [ ]:
# PROYECTO 3
# TODO: Tu implementación aquí

---
## PREGUNTAS DE REFLEXIÓN

Responde las siguientes preguntas basándote en tu experiencia con los ejercicios:

1. **Sobre el entrenamiento adversarial:**
   - ¿Por qué es importante balancear las actualizaciones de G y D?
   - ¿Qué pasa si D se vuelve demasiado bueno muy rápido?
   - ¿Cómo afecta el learning rate a la estabilidad?

2. **Sobre el espacio latente:**
   - ¿Qué observaste al interpolar entre dos puntos en z?
   - ¿Las transiciones son suaves o abruptas? ¿Por qué?
   - ¿Todas las dimensiones de z parecen igualmente importantes?

3. **Sobre mode collapse:**
   - ¿Experimentaste mode collapse en tus entrenamientos?
   - ¿Qué síntomas observaste?
   - ¿Qué técnicas ayudaron a mitigarlo?

4. **Sobre métricas:**
   - ¿IS y FID siempre concuerdan?
   - ¿Qué métricas consideras más útiles y por qué?
   - ¿Cómo se relacionan las métricas cuantitativas con la calidad visual?

5. **Sobre arquitecturas:**
   - ¿Qué diferencias notaste entre vanilla GAN y DCGAN?
   - ¿Por qué las convoluciones funcionan mejor para imágenes?
   - ¿Qué componentes arquitectónicos son más críticos?

## RECURSOS ADICIONALES

**Papers fundamentales:**
- Generative Adversarial Networks (Goodfellow et al., 2014)
- Unsupervised Representation Learning with DCGAN (Radford et al., 2015)
- Wasserstein GAN (Arjovsky et al., 2017)
- Progressive Growing of GANs (Karras et al., 2017)
- StyleGAN (Karras et al., 2019)

**Tutoriales recomendados:**
- https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html
- https://github.com/eriklindernoren/PyTorch-GAN

**Datasets para practicar:**
- MNIST (dígitos)
- Fashion-MNIST (ropa)
- CIFAR-10 (objetos)
- CelebA (rostros)
- LSUN (escenas)